# J.U.D.E. Tier 2 — QLoRA Fine-Tuning & Before/After Benchmark (`Qwen2.5-3B-Instruct` -> `J.U.D.E.-3B`)

This notebook fine-tunes `unsloth/Qwen2.5-3B-Instruct` using **4-bit QLoRA** on Google Colab Pro (T4/L4 GPU) for **J.U.D.E.**'s locked 18 Windows 11 + VMware Linux desktop agent schemas.

**Outputs for CL461 & Local Deployment:**
- Side-by-side **Before (Base `Qwen2.5-3B`) vs. After (`J.U.D.E.-3B`)** evaluation table & bar chart (`tier2_before_vs_after.png`, `tier2_before_vs_after.csv`, `tier2_metrics.json`)
- Training & validation loss curve (`tier2_loss_curve.png`)
- Quantized **`jude-3b-q4_k_m.gguf`** (~1.9 GB) + `Modelfile.jude_3b` ready for local Ollama

In [ ]:
# Cell 1: Verify Colab GPU and install Unsloth without upgrading Colab's preloaded numpy
import numpy as _np, subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))

!pip install -q uv
!uv pip install --system -q unsloth trl peft transformers accelerate bitsandbytes datasets "numpy=={_np.__version__}"

In [ ]:
# Cell 2: Load J.U.D.E. synthetic Tier 2 train & eval datasets
import json
import os
from pathlib import Path

REPO_DIR = Path('/content/JUDE')
if not REPO_DIR.exists():
    !git clone -q https://github.com/Baymax005/JUDE.git /content/JUDE
else:
    !git -C /content/JUDE pull -q

SCHEMAS_JSON = REPO_DIR / 'colab' / 'jude_tool_schemas.json'
TRAIN_JSONL = REPO_DIR / 'colab' / 'jude_agent_dataset.jsonl'
EVAL_JSONL = REPO_DIR / 'colab' / 'jude_agent_eval.jsonl'

# Auto-generate local JSONL splits using pure-Python OS-independent builder
if not TRAIN_JSONL.exists() and (REPO_DIR / 'colab' / 'build_tier2_dataset.py').exists():
    import subprocess, sys
    res = subprocess.run(
        [sys.executable, str(REPO_DIR / 'colab' / 'build_tier2_dataset.py')],
        capture_output=True,
        text=True,
    )
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'Dataset builder exited with {res.returncode}')

# Fallback if uploaded manually to /content
if not TRAIN_JSONL.exists() and Path('/content/jude_agent_dataset.jsonl').exists():
    SCHEMAS_JSON = Path('/content/jude_tool_schemas.json')
    TRAIN_JSONL = Path('/content/jude_agent_dataset.jsonl')
    EVAL_JSONL = Path('/content/jude_agent_eval.jsonl')

with SCHEMAS_JSON.open('r', encoding='utf-8') as f:
    schema_by_name = {s['function']['name']: s for s in json.load(f)}

def hydrate_tools(row):
    if 'tools' not in row and 'tool_names' in row:
        row['tools'] = [schema_by_name[n] for n in row['tool_names'] if n in schema_by_name]
    return row

with TRAIN_JSONL.open('r', encoding='utf-8') as f:
    jude_train_rows = [hydrate_tools(json.loads(line)) for line in f if line.strip()]
with EVAL_JSONL.open('r', encoding='utf-8') as f:
    jude_eval_rows = [hydrate_tools(json.loads(line)) for line in f if line.strip()]

print(f'Loaded {len(jude_train_rows)} J.U.D.E. training rows and {len(jude_eval_rows)} held-out eval rows.')

In [ ]:
# Cell 3: Load & blend public function-calling datasets (with HF_TOKEN fallback for gated xLAM)
import random
from datasets import Dataset, load_dataset

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        hf_token = ''

def format_qwen_tool_prompt(messages, tools):
    """Format chat + tools into Qwen2.5 ChatML with <tools> and <tool_call> tags."""
    sys_text = messages[0]['content'] if messages and messages[0]['role'] == 'system' else 'You are a helpful assistant.'
    tools_json = json.dumps(tools, ensure_ascii=False)
    sys_block = (
        f"{sys_text}\n\n# Tools\n"
        "You may call one or more functions to assist with the user query.\n"
        "You are provided with function signatures within <tools></tools> XML tags:\n"
        f"<tools>\n{tools_json}\n</tools>\n"
        "For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n"
        "<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call>"
    )
    chunks = [f"<|im_start|>system\n{sys_block}<|im_end|>"]
    for msg in messages:
        role = msg['role']
        if role == 'system':
            continue
        if role == 'assistant':
            tcs = msg.get('tool_calls') or []
            body_parts = []
            if msg.get('content'):
                body_parts.append(str(msg['content']).strip())
            for tc in tcs:
                fn = tc.get('function', tc)
                call_obj = {'name': fn['name'], 'arguments': fn.get('arguments', {})}
                body_parts.append(f"<tool_call>\n{json.dumps(call_obj, ensure_ascii=False)}\n</tool_call>")
            chunks.append(f"<|im_start|>assistant\n" + '\n'.join(body_parts) + "<|im_end|>")
        elif role == 'tool':
            chunks.append(f"<|im_start|>user\n<tool_response>\n{msg['content']}\n</tool_response><|im_end|>")
        else:
            chunks.append(f"<|im_start|>{role}\n{msg['content']}<|im_end|>")
    return '\n'.join(chunks)

formatted_texts = []
# Upsample J.U.D.E. domain rows (2x) so PC/Word/VMware/honest-refusal rules dominate
for _ in range(2):
    for row in jude_train_rows:
        formatted_texts.append({'text': format_qwen_tool_prompt(row['messages'], row['tools']), 'source': 'jude_os'})

# Public NousResearch/hermes-function-calling-v1 (ungated)
try:
    hermes = load_dataset('NousResearch/hermes-function-calling-v1', split='train[:900]')
    added_hermes = 0
    for ex in hermes:
        convs = ex.get('conversations') or []
        if not convs:
            continue
        parts = []
        for turn in convs:
            frm = turn.get('from', 'human')
            val = str(turn.get('value', '')).strip()
            role = 'system' if frm == 'system' else ('assistant' if frm in ('gpt', 'assistant') else 'user')
            parts.append(f'<|im_start|>{role}\n{val}<|im_end|>')
        formatted_texts.append({'text': '\n'.join(parts), 'source': 'hermes'})
        added_hermes += 1
    print(f'Loaded {added_hermes} rows from NousResearch/hermes-function-calling-v1.')
except Exception as exc:
    print(f'Hermes load skipped ({exc}); continuing with remaining sources.')

# Optional gated Salesforce/xlam-function-calling-60k when HF_TOKEN is configured
if hf_token:
    try:
        xlam = load_dataset('Salesforce/xlam-function-calling-60k', split='train[:1000]', token=hf_token)
        added_xlam = 0
        for ex in xlam:
            tools_obj = json.loads(ex['tools']) if isinstance(ex['tools'], str) else ex['tools']
            ans_obj = json.loads(ex['answers']) if isinstance(ex['answers'], str) else ex['answers']
            msgs = [
                {'role': 'system', 'content': 'You are a helpful tool-calling desktop assistant.'},
                {'role': 'user', 'content': ex['query']},
                {'role': 'assistant', 'content': '', 'tool_calls': [{'type': 'function', 'function': a} for a in ans_obj]},
            ]
            wrapped_tools = [{'type': 'function', 'function': t} for t in tools_obj]
            formatted_texts.append({'text': format_qwen_tool_prompt(msgs, wrapped_tools), 'source': 'xlam'})
            added_xlam += 1
        print(f'Loaded {added_xlam} rows from Salesforce/xlam-function-calling-60k.')
    except Exception as exc:
        print(f'xLAM load failed ({exc}); falling back cleanly to Hermes + J.U.D.E. OS dataset.')
else:
    print('HF_TOKEN not set -> skipping gated Salesforce/xlam-function-calling-60k (using Hermes + J.U.D.E. dataset).')

random.Random(42).shuffle(formatted_texts)
split_idx = int(len(formatted_texts) * 0.90)
train_ds = Dataset.from_list(formatted_texts[:split_idx])
val_ds = Dataset.from_list(formatted_texts[split_idx:])
print(f'Final blended split -> Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Cell 4: Load base unsloth/Qwen2.5-3B-Instruct in 4-bit NF4 and define evaluation helper
import re
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

_TC_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', flags=re.DOTALL)

def build_eval_prompt(row):
    sys_text = row['messages'][0]['content']
    user_text = row['messages'][1]['content']
    tools_json = json.dumps(row['tools'], ensure_ascii=False)
    sys_block = (
        f"{sys_text}\n\n# Tools\n"
        "You may call one or more functions to assist with the user query.\n"
        f"<tools>\n{tools_json}\n</tools>\n"
        "For each function call, return a json object within <tool_call></tool_call> tags.\n"
        "<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call>"
    )
    return f"<|im_start|>system\n{sys_block}<|im_end|>\n<|im_start|>user\n{user_text}<|im_end|>\n<|im_start|>assistant\n"

def evaluate_jude_model(eval_model, eval_tok, rows, label='Model'):
    FastLanguageModel.for_inference(eval_model)
    json_valid = 0
    tool_correct = 0
    arg_correct = 0
    refusal_total = 0
    refusal_honest = 0

    for idx, row in enumerate(rows):
        prompt = build_eval_prompt(row)
        inputs = eval_tok(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH - 256).to('cuda')
        with torch.no_grad():
            out_ids = eval_model.generate(
                **inputs,
                max_new_tokens=160,
                do_sample=False,
                pad_token_id=eval_tok.eos_token_id,
            )
        gen = eval_tok.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        exp_tool = row.get('expected_tool')
        exp_args = row.get('expected_args') or {}

        m = _TC_RE.search(gen)
        pred_tool = None
        pred_args = {}
        valid_json = False
        if m:
            try:
                obj = json.loads(m.group(1))
                pred_tool = obj.get('name')
                pred_args = obj.get('arguments') or {}
                valid_json = isinstance(pred_tool, str) and isinstance(pred_args, dict)
            except Exception:
                valid_json = False
        elif exp_tool is None:
            valid_json = True

        if valid_json:
            json_valid += 1
        if pred_tool == exp_tool:
            tool_correct += 1
            if exp_tool is None:
                arg_correct += 1
            else:
                ok_args = all(
                    k in pred_args and str(v).lower() in str(pred_args[k]).lower()
                    for k, v in exp_args.items() if v != ''
                )
                if ok_args:
                    arg_correct += 1

        if exp_tool is None:
            refusal_total += 1
            if pred_tool is None and 'alibaba' not in gen.lower() and 'qwen' not in gen.lower():
                refusal_honest += 1

    n = max(1, len(rows))
    metrics = {
        'label': label,
        'n_eval': len(rows),
        'json_valid_pct': round(100.0 * json_valid / n, 2),
        'tool_selection_pct': round(100.0 * tool_correct / n, 2),
        'arg_match_pct': round(100.0 * arg_correct / n, 2),
        'refusal_honesty_pct': round(100.0 * refusal_honest / max(1, refusal_total), 2),
    }
    print(metrics)
    return metrics

In [ ]:
# Cell 5: BEFORE Benchmark — Evaluate Base Qwen2.5-3B-Instruct before QLoRA
before_metrics = evaluate_jude_model(model, tokenizer, jude_eval_rows, label='Base Qwen2.5-3B')

In [ ]:
# Cell 6: Attach 4-bit QLoRA adapters (r=16) and fine-tune via SFTTrainer
import inspect
import math
import matplotlib.pyplot as plt
import pandas as pd
from transformers import TrainingArguments
from trl import SFTTrainer
try:
    from trl import SFTConfig
    ConfigCls = SFTConfig
except ImportError:
    ConfigCls = TrainingArguments

FastLanguageModel.for_training(model)
if hasattr(model, 'config'):
    model.config.use_cache = False

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

num_epochs = 3
batch_size = 2
grad_accum = 4
steps_per_epoch = max(1, math.ceil(len(train_ds) / (batch_size * grad_accum)))
warmup_steps = max(5, int(0.06 * steps_per_epoch * num_epochs))

valid_cfg_params = set(inspect.signature(ConfigCls.__init__).parameters)
raw_cfg = {
    'output_dir': '/content/jude_3b_checkpoints',
    'per_device_train_batch_size': batch_size,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': grad_accum,
    'num_train_epochs': num_epochs,
    'warmup_steps': warmup_steps,
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 15,
    'save_strategy': 'no',
    'fp16': not torch.cuda.is_bf16_supported(),
    'bf16': torch.cuda.is_bf16_supported(),
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'seed': 42,
    'report_to': 'none',
    'dataset_text_field': 'text',
    'max_seq_length': MAX_SEQ_LENGTH,
    'max_length': MAX_SEQ_LENGTH,
}
if 'eval_strategy' in valid_cfg_params:
    raw_cfg['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in valid_cfg_params:
    raw_cfg['evaluation_strategy'] = 'epoch'
cfg_kwargs = {k: v for k, v in raw_cfg.items() if k in valid_cfg_params}

trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
sft_kwargs = {
    'model': model,
    'train_dataset': train_ds,
    'eval_dataset': val_ds,
    'args': ConfigCls(**cfg_kwargs),
}
if 'tokenizer' in trainer_params:
    sft_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in trainer_params:
    sft_kwargs['processing_class'] = tokenizer
if 'dataset_text_field' in trainer_params and 'dataset_text_field' not in cfg_kwargs:
    sft_kwargs['dataset_text_field'] = 'text'
if 'max_seq_length' in trainer_params and 'max_seq_length' not in cfg_kwargs:
    sft_kwargs['max_seq_length'] = MAX_SEQ_LENGTH

trainer = SFTTrainer(**sft_kwargs)
train_stats = trainer.train()

OUT_REPORTS = Path('/content/jude_tier2_reports')
OUT_REPORTS.mkdir(parents=True, exist_ok=True)

train_logs = [x for x in trainer.state.log_history if 'loss' in x]
eval_logs = [x for x in trainer.state.log_history if 'eval_loss' in x]
plt.figure(figsize=(8, 4.5))
if train_logs:
    plt.plot([x['step'] for x in train_logs], [x['loss'] for x in train_logs], label='Train Loss', color='#0284C7')
if eval_logs:
    plt.plot([x['step'] for x in eval_logs], [x['eval_loss'] for x in eval_logs], 'o-', label='Val Loss', color='#F97316')
plt.title('J.U.D.E.-3B QLoRA Training & Validation Loss')
plt.xlabel('Optimizer Step')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_REPORTS / 'tier2_loss_curve.png', dpi=160)
plt.show()

In [ ]:
# Cell 7: AFTER Benchmark, Before-vs-After Chart, and Zip Report Bundle
import zipfile
import numpy as np

after_metrics = evaluate_jude_model(model, tokenizer, jude_eval_rows, label='J.U.D.E.-3B (QLoRA)')
comp_df = pd.DataFrame([before_metrics, after_metrics])
print('\n=== CL461 BEFORE vs AFTER COMPARISON ===')
display(comp_df)

comp_df.to_csv(OUT_REPORTS / 'tier2_before_vs_after.csv', index=False)
with (OUT_REPORTS / 'tier2_metrics.json').open('w', encoding='utf-8') as f:
    json.dump({'before': before_metrics, 'after': after_metrics}, f, indent=2)

metric_keys = ['json_valid_pct', 'tool_selection_pct', 'arg_match_pct', 'refusal_honesty_pct']
metric_labels = ['Valid JSON / Format', 'Tool Selection Acc', 'Arg Exact Match', 'Honest Refusal Rate']
x = np.arange(len(metric_labels))
width = 0.35
plt.figure(figsize=(9, 4.8))
plt.bar(x - width / 2, [before_metrics[k] for k in metric_keys], width, label='Base Qwen2.5-3B', color='#64748B')
plt.bar(x + width / 2, [after_metrics[k] for k in metric_keys], width, label='Fine-Tuned J.U.D.E.-3B (QLoRA)', color='#0EA5E9')
plt.ylim(0, 110)
plt.xticks(x, metric_labels, rotation=10)
plt.ylabel('Score (%)')
plt.title('J.U.D.E. Tier 2 Evaluation: Base Qwen2.5-3B vs QLoRA Fine-Tuned J.U.D.E.-3B')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_REPORTS / 'tier2_before_vs_after.png', dpi=160)
plt.show()

with zipfile.ZipFile('/content/jude_tier2_reports.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_REPORTS.iterdir():
        zf.write(p, arcname=p.name)
print('Saved /content/jude_tier2_reports.zip')

In [ ]:
# Cell 8: Export Q4_K_M GGUF (~1.9 GB) & Modelfile for Local Ollama
GGUF_DIR = Path('/content/jude_3b_gguf')
model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method='q4_k_m')

gguf_files = sorted(list(GGUF_DIR.glob('**/*.gguf')) + list(Path('/content').glob('*gguf*/*.gguf')))
q4_files = [p for p in gguf_files if 'q4_k_m' in p.name.lower()]
chosen_gguf = (q4_files or gguf_files or [GGUF_DIR / 'unsloth.Q4_K_M.gguf'])[0]
print('Exported GGUF(s):', gguf_files)
print('Selected GGUF:', chosen_gguf)

modelfile_txt = '''FROM ./__GGUF_NAME__

TEMPLATE """{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{- if .System }}
{{ .System }}
{{- end }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- else if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}"""

PARAMETER temperature 0.2
PARAMETER num_ctx 4096
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"

SYSTEM """
You are J.U.D.E., a local-first Windows 11 desktop AI agent with VMware Linux guest integration.
Always identify yourself strictly as J.U.D.E. (never as Qwen or Alibaba Cloud).
STRICT GROUNDING RULE: NEVER claim you opened, closed, typed, edited, sent a message, or saved anything unless you actually called a tool in this turn and received ok=true.
Be concise (1-2 sentences), natural, and do not repeat the user's name on every response.
"""
'''
modelfile_txt = modelfile_txt.replace('__GGUF_NAME__', chosen_gguf.name)
chosen_gguf.parent.mkdir(parents=True, exist_ok=True)
(chosen_gguf.parent / 'Modelfile.jude_3b').write_text(modelfile_txt, encoding='utf-8')
print(f'Wrote {chosen_gguf.parent / "Modelfile.jude_3b"}. Download the .gguf + Modelfile.jude_3b and run:')
print('  ollama create jude -f Modelfile.jude_3b')
print('  ollama rm qwen2.5:3b  # optional after verifying jude works')